# Model selection: immunogenicity tools: Nanobodies
This script aims to perform model selection on immunogenicity prediction tools. Previously different tools and settings have been tested. Comparision to anti-drug antibody (ADA) data using mainly spearman rank correlation, but also Mean Absolute Rank Error (MARE), to the tool scores has resulted in the tools and settings selected for testing in this script. Here the intention is to combine the tools to one predictive model, and perform ridge regression to investigate which tools have the most infulence on the best predicted response. Here ADA data will again be used as the correct/observed response varaible. In difference to previous testing the raw ADA score as well as the raw tool output score will be used in the model selection.\
\
The selected tools and settings for this secondary testing are\
1. netMHC1_EL_pep9:\
netMHCpan for MHC class 1, prediction type EL (elution ligand), peptide length 9, sliding window 5 and the only aviable pre prepare human allel panel with 27 alleles. These are the default settings for this tool.For this prediction 3 types of score are provided: the precentile_EL score, the immunogenicity score and the MHC pre processing score.\
2. netMHC1_pep14:\
This the same settings as scores above, expcet that the peptide length is 14. This are the best scoring settings for this tool\
3. netMHC_II_EL_pep12:\
netMHCpan for MHC class 2, prediciton type EL, peptide length 12, sliding window 5 and human 27 allele panel. For this tool there are two huaman allele panels, either 7 or 27. the human 27 allele panel scored marginally higher with all other combination of settings. Further, for this tool there are 2 different scores: the precentile_EL score and the immunogenicity score. THe MHC pre processing score is not availabe for these settings, only for peptide lengths 13 to 23. This tool and setting combination was the absolute highest scoring one in the initail testing\
4. netMHC_II_EL_pep15:\
This is the same settings as above (3.) but with peptide length 15. Here there are 3 scores: preceintile_EL, immunogenicity and MHC pre processing. This are the default settings for this tool.\
5. biophi_KabKabRelaxed:\
BioPhi OASis predicts how human a peptide is. The numbering scheme is Kabat, CDR definition Kabat and the human percent threshold is set to relaxed (>50%). This is the default setttings for this tool. This tool did not score very high on the spearman rank correlation in the initial testing, however that was calcualted on antibodies only and the main concern here is nanobodies. This tool had the best MARE scores for nanobodies and is therefore included here.\
6. biophi_KabKabStrict\
BioPhi OASis predicts how human a peptide is. The numbering scheme is Kabat, CDR definition Kabat and the human percent threshold is set to strict (>10%). This is the highest scoring settings for this tool.\
7. waltz\
This is a aggregation prediciton tool that has been included because it might improve the immunogenicity predicitons. Default settings are used.\

In [97]:
# load libaries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [98]:
netMHC1_EL_pep9 = pd.read_csv("tool_outputs/NB_netMHC_peplen9_18_03_2026.csv")
netMHC1_EL_pep14 = pd.read_csv("tool_outputs/NB_netMHC_peplen14_18_03_2026.csv")
netMHC_II_EL_pep12 = pd.read_csv("tool_outputs/NB_netMHC_II_peplen12_18_03_2026.csv")
netMHC_II_EL_pep15 = pd.read_csv("tool_outputs/NB_netMHC_II_peplen15_18_03_2026.csv")
biophi_KabKabRelaxed = pd.read_excel("tool_outputs/NB_NrKab_CDRKab_relaxed_18_03_2026.xlsx")
biophi_KabKabStrict = pd.read_excel("tool_outputs/NB_NrKab_CDRKab_strict_18_03_2026.xlsx")
waltz = pd.read_csv("tool_outputs/NB_waltz.txt", sep="\t", header=None, names=["antibody", "waltz_score", "..."])

# load table with the antibody names
seqTable = pd.read_csv("tool_outputs/NB_seqTable_netMHC_II_peplen12.csv")

# load file containing the antibody name and their observed ADA percantage
ADA = pd.read_csv("../../../data/ADA_6NB.txt", sep="\t", header=None, names=["antibody", "ADA_percentage"])

Some of the nanobodies have 2 chains. BioPhi reads them as two different nanobodies, so they need to be merged. This is done below.

In [99]:
# First filter so there is only the two columns of interest
# Then remove everything after the last underscore, then the antibody names will be the same
# Then merge the rows with the same antibody names, for the OASis identity take the mean value for the new row.
biophi_KabKabRelaxed = (
    biophi_KabKabRelaxed[['Antibody', 'OASis Identity']]
    .assign(Antibody=lambda df: df['Antibody'].str.replace(r'_[^_]*$', '', regex=True))
    .groupby('Antibody', as_index=False)['OASis Identity']
    .mean()
    .rename(columns={'Antibody':'antibody'}) # to make merging easier later
    .rename(columns={'OASis Identity':'biophi_KabKabRelaxed_score'}) # to make merging easier later
)


# Same for the other df
biophi_KabKabStrict = (
    biophi_KabKabStrict[['Antibody', 'OASis Identity']]
    .assign(Antibody=lambda df: df['Antibody'].str.replace(r'_[^_]*$', '', regex=True))
    .groupby('Antibody', as_index=False)['OASis Identity']
    .mean()
    .rename(columns={'Antibody':'antibody'}) # to make merging easier later
    .rename(columns={'OASis Identity':'biophi_KabKabStrict_score'}) # to make merging easier later
)


In [100]:
# Sanity checks

# Check nr of antibodies
if netMHC1_EL_pep14['seq #'].nunique()!=6:
        print( "netMHC1_EL_pep14 does not have 6 antibodies")
if netMHC1_EL_pep9['seq #'].nunique()!=6:
        print( "netMHC1_EL_pep9 does not have 6 antibodies")
if netMHC_II_EL_pep12['seq #'].nunique()!=6:
        print( "netMHC_II_EL_pep12 does not have 6 antibodies")   
if netMHC_II_EL_pep15['seq #'].nunique()!=6:
        print( "netMHC_II_EL_pep15 does not have 6 antibodies")
if biophi_KabKabRelaxed['antibody'].nunique()!=6:
        print( "biophi_KabKabRelaxed does not have 6 antibodies")
if biophi_KabKabStrict['antibody'].nunique()!=6:
        print( "biophi_KabKabStrict does not have 6 antibodies")
if waltz['antibody'].nunique()!=6:
        print( "waltz does not have 6 antibodies")

In [101]:
# the netMHC tools do not have the antibody name in the output, only a sequence number
# Here the seq # will be mapped back to the original antibody name using the seqTable
netMHC1_EL_pep9 = netMHC1_EL_pep9.merge(seqTable[['seq #', 'sequence name']], how='left')
netMHC1_EL_pep14 = netMHC1_EL_pep14.merge(seqTable[['seq #', 'sequence name']], how='left')
netMHC_II_EL_pep12 = netMHC_II_EL_pep12.merge(seqTable[['seq #', 'sequence name']], how='left')
netMHC_II_EL_pep15 = netMHC_II_EL_pep15.merge(seqTable[['seq #', 'sequence name']], how='left')

# Compute scores

netMHCpan tools give one score for each peptide - HLA allele combination
Therefore, for each nanobody a immunogenicity score will be calculated. The definition of what score is considerd immunogenetic differs for the different tools \
for each netMHC score of interest: precentile score, immunogenicity socre and pre processing score

In [102]:
# netMHC1_EL_pep9 percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep9_percentile = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_percentile')
    )

# netMHC1_EL_pep9 Immunogenicity score 

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep9_immunogenicity_score = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_immunogenicity_score')
    )

# netMHC1_EL_pep9 Preprocessing score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMCH1_pep9_preProcess = netMHC1_EL_pep9.groupby('sequence name')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep9_preProcess'})

# netMHC1_EL_pep14

# Percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep14_percentile = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_percentile')
    )

# Immunogenicity score

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep14_immunogenicity_score = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_immunogenicity_score')
    )

# Pre-proocessing score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 

netMCH1_pep14_preProcess = netMHC1_EL_pep14.groupby('sequence name')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep14_preProcess'})


# netMHC_II_EL_pep12

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep12_percentile = (
    netMHC_II_EL_pep12.assign(immunogenic=netMHC_II_EL_pep12['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep12_percentile')
    )

# Immunogenicity score

# remove the rows with the immunogenicity score of '-' before calculating the mean
netMHC_II_EL_pep12 = netMHC_II_EL_pep12[netMHC_II_EL_pep12['immunogenicity score'] != '-']
# make the column with the immunogenicity score into a numeric column
netMHC_II_EL_pep12['immunogenicity score'] = pd.to_numeric(netMHC_II_EL_pep12['immunogenicity score'])

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody.
netMHC_II_pep12_immunogenicity_score = netMHC_II_EL_pep12.groupby('sequence name')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep12_immunogenicity_score'})


# Does not have pre processing score. Due to tool not being able to predict pre processing on peptides shorter than 13 amino acids


# netMHC_II_EL_pep15

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep15_percentile = (
    netMHC_II_EL_pep15.assign(immunogenic=netMHC_II_EL_pep15['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep15_percentile')
    )

# Immunogenicity score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_immunogenicity_score = netMHC_II_EL_pep15.groupby('sequence name')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep15_immunogenicity_score'})

# Pre-proocessing score
# MHC class 2 has 2 preprocessing scores of interest: mhcii-np cleavage probability score and mhcii-np cleavage probability percentile rank

# mhcii-np cleavage probability

# remove the rows with the cleavage probability score of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability score'] != '-']
# make the column with the cleavage probability score into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability score'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability score'])
# Compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProb = netMHC_II_EL_pep15.groupby('sequence name')['mhcii-np cleavage probability score'].mean().reset_index().rename(columns={'processing total score': 'netMHC_II_pep15_preProcess_cleavProb'})

# mhcii-np cleavage probability percentile rank

# remove the rows with the cleavage probability percentile rank of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] != '-']
# make the column with the cleavage probability percentile rank into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'])
# compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProbPercentile = netMHC_II_EL_pep15.groupby('sequence name')['mhcii-np cleavage probability percentile rank'].mean().reset_index().rename(columns={'mhcii-np cleavage probability percentile rank': 'netMHC_II_pep15_preProcess_cleavProbPercentile'})

# rename the column '...' to nr_aggs
waltz = waltz.rename(columns={'...': 'nr_aggs'})

# compute the total number of amio acids that are considerd immunogenetic
def sum_ranges(s):
    if pd.isna(s):
        return 0
    total = 0
    for part in s.split(';'):
        start, end = part.strip().split('-')
        total += int(end) - int(start) + 1 # beacuse the values are inclusive
    return total

waltz['nr_aggs'] = waltz['waltz_score'].apply(sum_ranges)

In [103]:
waltz

,antibody,waltz_score,nr_aggs
0,Gontivimab_ALX-0171,28-37; 47-60; 91-96; 105-114; 154-163; 173-186...,80
1,Caplacizumab,91-97,7
2,Vobarilizumab,27-37; 48-53; 90-101; 212-220,38
3,Tarperprumig_ALXN1820,33-37; 47-54; 90-101; 213-223,36
4,2Rs15d,25-32; 90-106,25
5,Ozoralizumab,47-61; 91-97; 206-214,31


In [104]:
# create df with all the dfs that shall me merged into one
dfs_to_merge = [
    netMHC1_pep9_percentile,
    netMHC1_pep9_immunogenicity_score,
    netMCH1_pep9_preProcess,
    netMHC1_pep14_percentile,
    netMHC1_pep14_immunogenicity_score,
    netMCH1_pep14_preProcess,
    netMHC_II_pep12_percentile,
    netMHC_II_pep12_immunogenicity_score,
    netMHC_II_pep15_percentile,
    netMHC_II_pep15_immunogenicity_score,
    netMHC_II_pep15_preProcess_cleavProb,
    netMHC_II_pep15_preProcess_cleavProbPercentile,
]

# Rename the sequence name column to antibody to make the merging easier. 
dfs_to_merge = [df.rename(columns={'sequence name': 'antibody'}) for df in dfs_to_merge]

# Create the df with all scores, start with the ADA scores
all_predictors_andADA = ADA

# Merge all dfs on antibody name
for df in dfs_to_merge:
    all_predictors_andADA = all_predictors_andADA.merge(df, on='antibody', how='left')

# remove all whitespaces in antobdy column
all_predictors_andADA['antibody'] = all_predictors_andADA['antibody'].str.strip()

# Then merge the three last dfs
all_predictors_andADA = all_predictors_andADA.merge(waltz[['antibody','nr_aggs']], on='antibody', how='left').rename(columns={'nr_aggs': 'waltz_nr_aggs'})
all_predictors_andADA = all_predictors_andADA.merge(biophi_KabKabRelaxed, on='antibody', how='left')
all_predictors_andADA = all_predictors_andADA.merge(biophi_KabKabStrict, on='antibody', how='left')

all_predictors_andADA

,antibody,ADA_percentage,netMHC1_pep9_percentile,netMHC1_pep9_immunogenicity_score,netMHC1_pep9_preProcess,netMHC1_pep14_percentile,netMHC1_pep14_immunogenicity_score,netMHC1_pep14_preProcess,netMHC_II_pep12_percentile,netMHC_II_pep12_immunogenicity_score,netMHC_II_pep15_percentile,netMHC_II_pep15_immunogenicity_score,mhcii-np cleavage probability score,netMHC_II_pep15_preProcess_cleavProbPercentile,waltz_nr_aggs,biophi_KabKabRelaxed_score,biophi_KabKabStrict_score
0,Ozoralizumab,2.60,3.069736,41.891892,-3.441872,0.238949,32.258065,-3.580015,7.070707,98.946795,7.491582,95.393005,0.085804,33.565000,31,0.822430,0.607477
1,Caplacizumab,11.00,3.549383,53.333333,-3.381630,0.193237,60.000000,-3.505279,11.419753,98.935075,10.466989,94.663083,0.121135,32.588182,7,0.608333,0.391667
2,2Rs15d,20.00,2.596054,41.121495,-3.471023,0.108932,47.058824,-3.568047,5.291005,99.488019,4.409171,95.527548,0.137821,23.978421,25,0.392523,0.168224
3,Gontivimab_ALX-0171,34.10,2.944748,61.065574,-3.475876,0.247947,61.924686,-3.606141,9.750567,99.351057,7.716049,95.516623,0.155020,23.435319,80,0.457627,0.169492
4,Vobarilizumab,41.00,2.842755,41.228070,-3.435665,0.298954,34.080717,-3.571817,7.160494,99.149196,6.748971,95.134060,0.151941,22.027045,38,0.752667,0.550037
5,Tarperprumig_ALXN1820,69.12,3.355492,44.444444,-3.359280,0.226427,44.978166,-3.496130,8.510638,99.192289,9.903382,93.943080,0.133803,23.269556,36,0.593106,0.508931
